
User Based Collaborative Filtering (Kullanıcı Tabanlı İşbirlikçi Filtreleme) algoritması, öneri sistemlerinde kullanıcıların geçmişteki davranışlarını (örneğin puanladıkları filmler) baz alarak benzer kullanıcıları bulur ve onların beğendiği ürünleri/filmleri önerir.

Bu yöntemin temel amacı, bir kullanıcının beğenebileceği bir ürünü, ona en çok benzeyen kullanıcıların tercihlerini inceleyerek tahmin etmektir.

Nasıl çalışır?
1. Her kullanıcının ürünlere verdiği puanlar üzerinden kullanıcılar arasındaki benzerlik (örneğin korelasyon veya kosinüs benzerliği) hesaplanır.
2. Hedef kullanıcının en çok benzediği kullanıcılar seçilir.
3. Benzer kullanıcıların yüksek puan verdiği fakat hedef kullanıcının henüz deneyimlemediği ürünler önerilir.

Avantajları:
- Popüler olmayan veya yeni ürünler için de etkili öneriler sunabilir.
- Kişiselleştirilmiş tavsiyeler verir.

Dezavantajları:
- Kullanıcı veya ürün sayısı azsa başlangıçta öneri yapmak zor olabilir (soğuk başlangıç problemi).
- Çok büyük veri setlerinde hesaplama maliyeti yüksektir.
- Seyrek veri durumunda (kullanıcılar az filme puan verdiyse) benzer kullanıcı bulmak zorlaşır.


# İş Problemi

Online bir film izleme platformu (örneğin kuzukuzu.tv) daha önce hazırlamış olduğu tavsiye sistemini geliştirmek istemektedir.

İçerik tabanlı öneri sistemlerini ve item-based öneri sistemlerini deneyen şirket, kullanıcılara DAHA FAZLA ÖZELLEŞTİRME yapılmasını istemektedir.

Filmler özelinde benzer beğenilme yapılarına göre öneriler yapılmış fakat bu genel önerileri kullanıcıların kullanıcılar benzerliği üzerinden daha fazla özelleştirmek istemektedirler.

### Adım 1: Veri Setinin Hazırlanması


In [1]:
def create_user_movie_df():
    """
    Kullanıcı-film pivot tablosunu oluşturan fonksiyon.
    movie.csv ve rating.csv dosyalarını okur, user-movie rating matrisini döndürür.
    """
    import pandas as pd
    # Film bilgilerini oku
    movie = pd.read_csv('datasets/movie_lens_dataset/movie.csv')
    # Puan bilgilerini oku
    rating = pd.read_csv('datasets/movie_lens_dataset/rating.csv')
    # İki veri setini birleştir
    df = movie.merge(rating, how="left", on="movieId")
    # Filmlerin puan sayısını hesapla
    comment_counts = df["title"].value_counts().reset_index()
    comment_counts.columns = ["title", "count"]
    # Nadir filmleri belirle (1000'den az yorumu olanlar)
    rare_movies = comment_counts[comment_counts["count"] <= 1000]["title"]
    # Yaygın filmleri seç
    common_movies = df[~df["title"].isin(rare_movies)]
    # Kullanıcı-film puan matrisini oluştur
    user_movie_df = common_movies.pivot_table(index="userId", columns="title", values="rating")
    return user_movie_df

user_movie_df = create_user_movie_df()

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.expand_frame_repr', False)
# user_movie_df tablosunda bulunan kullanıcılardan rastgele bir kullanıcının indeks numarasını seçmek için:
# .sample(1, random_state=45) fonksiyonu ile indexlerden rastgele birini seçiyoruz.
# .iloc[0] ise seçilen değeri doğrudan getirir (seri içinden saf değer olarak).
random_user = pd.Series(user_movie_df.index).sample(1, random_state=45).iloc[0]
random_user

np.float64(28941.0)

### Adım 2: Öneri Yapılacak Kullanıcının İzlediği Filmlerin Belirlenmesi


In [3]:
# random_user_df: Rastgele seçilmiş kullanıcının (random_user) izlediği filmleri ve bu filmlere verdiği puanları içeren satırı user_movie_df tablosundan çekiyoruz.
random_user_df = user_movie_df[user_movie_df.index == random_user]

# movies_watched: random_user'ın izlediği (yani puan girdiği) filmlerin isimlerini içeren bir liste oluşturuyoruz.
movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()

# Belirli bir filmin (örneğin "Silence of the Lambs, The (1991)") bu kullanıcı tarafından izlenip izlenmediğini ve verdiği puanı kontrol ediyoruz.
user_movie_df.loc[user_movie_df.index == random_user,
                  user_movie_df.columns == "Silence of the Lambs, The (1991)"]


title,"Silence of the Lambs, The (1991)"
userId,
28941.0,1.0


In [4]:
len(movies_watched)


33

### Adım 3: Aynı Filmleri İzleyen Diğer Kullanıcıların Verisine ve Id'lerine Erişmek


In [5]:
# Rastgele seçilen kullanıcının izlediği filmlerin sütunları, user_movie_df tablosundan alınarak yeni bir DataFrame oluşturulur.
# Yani, random_user'ın izlediği tüm filmlere ait puanları, tüm kullanıcılar için altset olarak seçmiş oluruz.
movies_watched_df = user_movie_df[movies_watched]

# Bu oluşturulan DataFrame'in boyutunu (satır ve sütun sayısı) görüntüleyerek kaç kullanıcı ve kaç film olduğu kontrol edilir.
movies_watched_df.shape


(138493, 33)

In [6]:
# Kullanıcıların, random_user'ın izlediği filmlerden kaç tanesini izlediğini buluyoruz.
user_movie_count = movies_watched_df.notnull().sum(axis=1)

# Sonucu DataFrame olarak kullanıcı id ve izlenen film sayısı şeklinde düzenliyoruz.
user_movie_count = user_movie_count.reset_index()
user_movie_count.columns = ["userId", "movie_count"]

# 20'den fazla ortak film izleyen kullanıcıları, izlenen film sayısına göre azalan şekilde sıralıyoruz.
user_movie_count[user_movie_count["movie_count"] > 20].sort_values("movie_count", ascending=False)


,userId,movie_count
94230,94231.0,33
100398,100399.0,33
118204,118205.0,33
15918,15919.0,33
124051,124052.0,33
...,...,...
79214,79215.0,21
79174,79175.0,21
9105,9106.0,21
78515,78516.0,21


In [7]:
# Burada, rasgele seçilmiş kullanıcının izlediği filmlerden en az 20 tanesini izlemiş olan diğer kullanıcıların userId'lerini seçiyoruz.
users_same_movies = user_movie_count[user_movie_count["movie_count"] > 20]["userId"]

# Alternatif olarak, ortak izlenen film sayısı olarak daha dinamik bir eşik kullanmak isteyebiliriz.
# Örneğin, rasgele kullanıcının izlediği filmlerin %60'ından fazlasını izlemiş olan kullanıcılar:
# users_same_movies = user_movie_count[user_movie_count["movie_count"] > perc]["userId"]
# perc = len(movies_watched) * 60 / 100


### Adım 4: Öneri Yapılacak Kullanıcı ile En Benzer Davranışlı Kullanıcıların Belirlenmesi

In [8]:
# Bu satırda, hem rasgele seçilmiş kullanıcının izlediği filmleri hem de aynı filmleri izleyen diğer kullanıcıların verilerini birleştiriyoruz.
# Öncelikle, movies_watched_df içinden yalnızca users_same_movies listesinde yer alan kullanıcıların verilerini seçiyoruz.
# Ardından, random_user_df[movies_watched] ifadesi ile rasgele seçilmiş kullanıcının aynı filmlere ait verisi alınır.
# İki DataFrame'i pd.concat ile alt alta birleştirip final_df adlı yeni bir DataFrame oluşturuyoruz.
# Bu DataFrame, hem hedef kullanıcının hem de onunla benzer film geçmişine sahip kullanıcıların verilerini içerir ve analiz için kullanılır.
final_df = pd.concat([movies_watched_df[movies_watched_df.index.isin(users_same_movies)],
                      random_user_df[movies_watched]])
final_df

title,Ace Ventura: Pet Detective (1994),Ace Ventura: When Nature Calls (1995),Aladdin (1992),"American President, The (1995)",Apollo 13 (1995),Babe (1995),Bullets Over Broadway (1994),Clueless (1995),Disclosure (1994),Forrest Gump (1994),Four Weddings and a Funeral (1994),Home Alone (1990),Jurassic Park (1993),Like Water for Chocolate (Como agua para chocolate) (1992),Little Women (1994),Mr. Holland's Opus (1995),Mrs. Doubtfire (1993),Much Ado About Nothing (1993),Muriel's Wedding (1994),Nine Months (1995),Operation Dumbo Drop (1995),"Piano, The (1993)","Postman, The (Postino, Il) (1994)",Ready to Wear (Pret-A-Porter) (1994),"Remains of the Day, The (1993)",Sabrina (1995),Schindler's List (1993),"Secret Garden, The (1993)",Sense and Sensibility (1995),Shadowlands (1993),"Silence of the Lambs, The (1991)",Star Trek: Generations (1994),Stargate (1994)
userId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
130.0,4.0,3.0,NaN,3.0,3.0,NaN,NaN,3.0,5.0,5.0,5.0,3.0,4.0,3.0,NaN,4.0,4.0,NaN,3.0,NaN,NaN,4.0,4.0,NaN,3.0,NaN,5.0,NaN,NaN,3.0,5.0,NaN,3.0
156.0,3.0,NaN,NaN,5.0,5.0,3.0,NaN,NaN,4.0,5.0,3.0,4.0,5.0,NaN,NaN,4.0,5.0,4.0,3.0,3.0,4.0,5.0,NaN,NaN,NaN,4.0,5.0,NaN,4.0,4.0,5.0,3.0,4.0
158.0,2.0,1.0,4.0,4.0,3.0,5.0,NaN,4.0,NaN,5.0,4.0,3.0,3.0,NaN,NaN,3.0,3.0,NaN,5.0,3.0,3.0,NaN,5.0,NaN,5.0,3.0,5.0,5.0,4.0,5.0,5.0,NaN,NaN
184.0,2.0,3.0,3.0,4.0,4.0,NaN,3.0,NaN,4.0,3.0,3.0,3.0,5.0,NaN,5.0,4.0,5.0,5.0,NaN,3.0,3.0,NaN,NaN,NaN,4.0,4.0,5.0,4.0,NaN,4.0,5.0,3.0,4.0
295.0,NaN,NaN,3.0,3.0,3.0,3.0,3.0,2.0,NaN,4.0,3.0,3.0,3.0,3.0,NaN,3.0,3.0,3.0,3.0,NaN,NaN,5.0,NaN,NaN,3.0,3.0,4.0,3.0,4.0,NaN,4.0,3.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138279.0,3.0,NaN,NaN,3.0,5.0,5.0,5.0,4.0,NaN,5.0,4.0,2.0,5.0,5.0,5.0,4.0,3.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,5.0,NaN,5.0,NaN,4.0,NaN,5.0,3.0,3.0
138382.0,1.0,1.0,4.0,2.0,3.0,5.0,NaN,4.0,3.0,5.0,3.0,3.0,5.0,5.0,NaN,NaN,5.0,NaN,NaN,3.0,NaN,1.0,NaN,NaN,NaN,3.0,NaN,3.0,3.0,NaN,5.0,NaN,4.0
138415.0,1.0,NaN,3.0,4.0,3.0,5.0,NaN,NaN,NaN,4.0,5.0,NaN,3.0,3.0,3.0,4.0,4.0,NaN,4.0,4.0,NaN,3.0,5.0,NaN,5.0,NaN,5.0,3.0,4.0,4.0,NaN,3.0,NaN


In [ ]:
import numpy as np

# Burada, öneri yapılacak kullanıcı ve onunla yüksek sayıda ortak film izlemiş kullanıcıların film derecelendirme verilerinden oluşan final_df üzerinden,
# kullanıcılar arası davranış benzerliğini ölçmek amacıyla korelasyon matrisini oluşturuyoruz.
# .T (transpose) işlemi ile filmleri satırlardan sütunlara geçiriyoruz; böylelikle aynı filmler sütun, kullanıcılar satır olacak ve
# kullanıcılar arası korelasyon için .corr() fonksiyonu doğrudan çalışacak hale geliyor.
corr_matrix = final_df.T.corr()

# Kullanıcı çiftlerinin korelasyonunu uzun formatta bir DataFrame'e (long-form) manuel olarak aktarıyoruz.
# Bunun nedeni: Eğer DataFrame'in sütun isimlerinde tekrar eden (duplicate) başlıklar olursa, pandas'ın .stack() fonksiyonu hata verebilir.
# Bu yüzden tüm kullanıcı çiftleri için korelasyonları bir döngü ile teker teker topluyoruz.
corr_pairs = []
index_array = corr_matrix.index.to_numpy()  # Korelasyon matrisindeki tüm kullanıcıların ID'lerini bir NumPy dizisine dönüştürüyoruz.

# İki döngüyle tüm kullanıcı çiftlerini geziyoruz fakat i+1'den başladığımızdan yalnızca üst üçgen (her çifti bir kez) alınıyor.
for i in range(len(index_array)):
    for j in range(i + 1, len(index_array)):
        user_id_1 = index_array[i]
        user_id_2 = index_array[j]
        corr_value = corr_matrix.iloc[i, j]  # Bu iki kullanıcı arasındaki korelasyon değerini alıyoruz.
        # Eğer korelasyon "nan" değilse (yani iki kullanıcıda ortak derecelendirilmiş film varsa), değeri listeye ekliyoruz.
        if not np.isnan(corr_value):
            corr_pairs.append((user_id_1, user_id_2, corr_value))

# Elde edilen kullanıcı çiftleri ve korelasyonlarını bir DataFrame'e aktarıyoruz.
corr_df = pd.DataFrame(corr_pairs, columns=['user_id_1', 'user_id_2', 'correlation'])

# Son olarak, bu kullanıcı çiftlerini korelasyon değerine göre azalan biçimde sıralıyoruz.
# Sıralama sayesinde, en üstte birbiriyle en benzer davranış gösteren kullanıcı çiftleri yer alacak.
# drop_duplicates() fonksiyonu ek bir temizlik adımı olarak burada kalmış olabilir; üst üçgenden aldığımız için tekrar yok, yine de olası tekrarları silmiş oluyoruz.
corr_df = corr_df.sort_values('correlation', ascending=False).drop_duplicates()

In [11]:
corr_df

,user_id_1,user_id_2,correlation
2008974,28941.0,28941.0,1.000000
4130138,75993.0,82143.0,1.000000
1187318,16562.0,41005.0,1.000000
2194301,32344.0,68063.0,0.977387
2279102,34350.0,96952.0,0.971667
...,...,...,...
2991916,48416.0,117826.0,-0.883969
1485062,21398.0,62575.0,-0.896612
2261996,34103.0,80593.0,-0.898718
2419639,37121.0,60562.0,-0.915003


In [12]:
corr_df =corr_df.reset_index()

In [15]:

# 'corr' isminde bir sütun yok, doğru sütun adı 'correlation' olduğu için filtrelemeyi bu şekilde yapıyoruz.
# İlk olarak, corr_df içerisinde random_user olan ve korelasyon değeri 0.65 ve üzeri olan kullanıcı çiftlerini seçiyoruz.
# Sadece 'user_id_2' ve 'correlation' sütunlarını alıp, index sıfırlıyoruz.
top_users = corr_df[
    (corr_df["user_id_1"] == random_user) & (corr_df["correlation"] >= 0.65)
][["user_id_2", "correlation"]].reset_index(drop=True)

# Ardından, ikinci kullanıcıya (user_id_2) olan korelasyon değerine göre büyükten küçüğe sıralama yapıyoruz.
top_users = top_users.sort_values(by='correlation', ascending=False)

# Son olarak, 'user_id_2' sütununu 'userId' olarak yeniden adlandırıyoruz.
top_users.rename(columns={"user_id_2": "userId"}, inplace=True)

# Elde edilen top_users DataFrame'i, random_user ile yüksek korelasyon (benzer zevk) gösteren kullanıcıları ve korelasyon değerlerini içerir.

In [16]:
top_users

,userId,correlation
0,28941.0,1.000000
1,45158.0,0.800749
2,101628.0,0.790405
3,127259.0,0.763925
4,128241.0,0.752587
5,93089.0,0.738536
6,136259.0,0.737177
7,117826.0,0.736011
8,58769.0,0.734092
9,67346.0,0.732943


In [17]:
# Şimdi öneri yapılacak kullanıcımızla yüksek korelasyon gösteren kullanıcıların (top_users) puanlamalarını getiriyoruz.
# Bunun için ana 'rating' verisini içe aktarıp, yalnızca bu kullanıcılar ve verdikleri puanları ile birleştiriyoruz.
# Böylece hem userId, hem verdikleri film puanları, hem de random_user ile olan korelasyon değerleri tek bir tabloda olacak.

rating = pd.read_csv('datasets/movie_lens_dataset/rating.csv')  # Puanlama datasını yüklüyoruz.

# top_users tablosundaki kullanıcılarla rating tablosunu birleştiriyoruz.
top_users_ratings = top_users.merge(
    rating[["userId", "movieId", "rating"]],  # Sadece ihtiyacımız olan sütunları seçiyoruz.
    how='inner'
)

# Kendi kendimizi (random_user) önerilerden hariç tutmak için o kullanıcıyı çıkarıyoruz.
top_users_ratings = top_users_ratings[top_users_ratings["userId"] != random_user]

top_users_ratings

,userId,correlation,movieId,rating
33,45158.0,0.800749,1,1.5
34,45158.0,0.800749,3,1.0
35,45158.0,0.800749,17,3.5
36,45158.0,0.800749,19,2.0
37,45158.0,0.800749,22,3.0
...,...,...,...,...
13964,82666.0,0.655212,593,4.0
13965,82666.0,0.655212,594,4.0
13966,82666.0,0.655212,595,3.0
13967,82666.0,0.655212,597,3.0


### Adım 5: Weighted Average Recommendation Score'un Hesaplanması


In [ ]:
# Öncelikle, yüksek korelasyona sahip kullanıcıların (top_users) verdikleri film puanlarını ve random_user ile olan korelasyonlarını içeren dataframe üzerinden
# her kullanıcının film puanını o kullanıcı ile random_user arasındaki korelasyon değeriyle çarpıyoruz. Böylece ağırlıklı bir puan elde ediyoruz.
top_users_ratings['weighted_rating'] = top_users_ratings['correlation'] * top_users_ratings['rating']

# Her film (movieId) için ağırlıklı ortalama (weighted_rating'in ortalaması) hesaplanıyor.
# İlk satır çıktıya etki etmiyor, sadece işlemi gösteriyor.
top_users_ratings.groupby('movieId').agg({"weighted_rating": "mean"})

# Her film için ağırlıklı ortalamalar yeni bir dataframe'de tutuluyor.
recommendation_df = top_users_ratings.groupby('movieId').agg({"weighted_rating": "mean"})

# Film id'lerini index olmaktan çıkarıp normal sütun haline getiriyoruz.
recommendation_df = recommendation_df.reset_index()

# Sadece ağırlıklı puanı 3.5'ten büyük olan filmleri filtreliyoruz. Bu, random_user'a önerilecek filmler için alt limitimiz.
recommendation_df[recommendation_df["weighted_rating"] > 3.5]

# Son olarak, önerilecek filmleri ağırlıklı puanlarına göre azalan şekilde sıralıyoruz ve "movies_to_be_recommend" olarak saklıyoruz.
movies_to_be_recommend = recommendation_df[recommendation_df["weighted_rating"] > 3.5].sort_values("weighted_rating", ascending=False)

# Filmlerin isimlerini gösterebilmek için film datasını yüklüyoruz.
movie = pd.read_csv('datasets/movie_lens_dataset/movie.csv')

# Sonuç olarak, önerilecek filmlere film başlıklarını ekleyerek gösteriyoruz.
movies_to_be_recommend.merge(movie[["movieId", "title"]])


,movieId,weighted_rating,title
0,53,3.952023,Lamerica (1994)
1,326,3.857478,To Live (Huozhe) (1994)
2,887,3.680055,Talk of Angels (1998)
3,1893,3.680055,Beyond Silence (Jenseits der Stille) (1996)
4,2675,3.680055,Twice Upon a Yesterday (a.k.a. Man with Rain i...
5,2883,3.680055,Mumford (1999)
6,3179,3.680055,Angela's Ashes (1999)
7,1184,3.670461,Mediterraneo (1991)
8,6837,3.664714,Love Affair (1939)
9,32234,3.664714,Julia (1977)


### Adım 6: Çalışmanın Fonksiyonlaştırılması

In [ ]:
def create_user_movie_df():
    """
    Kullanıcı-film matrisini oluşturan fonksiyondur.
    
    Returns:
        user_movie_df (pd.DataFrame): Kullanıcıların yaygın filmlere (rare olmayan) verdiği puanlardan oluşan kullanıcı-film pivot tablosu.
    """
    import pandas as pd
    # Filmler ve derecelendirmeler veri setleri okunuyor.
    movie = pd.read_csv('datasets/movie_lens_dataset/movie.csv')
    rating = pd.read_csv('datasets/movie_lens_dataset/rating.csv')
    # Filmler ile derecelendirmeler birleştiriliyor.
    df = movie.merge(rating, how="left", on="movieId")
    # Her filmin aldığı toplam puan/yorum sayısı sayılıyor.
    comment_counts = pd.DataFrame(df["title"].value_counts())
    # 1000 yorumdan az alan filmler nadir (rare) olarak kabul ediliyor.
    rare_movies = comment_counts[comment_counts["title"] <= 1000].index
    # Sadece yaygın filmler filtreleniyor.
    common_movies = df[~df["title"].isin(rare_movies)]
    # Kullanıcı-film pivot tablosu oluşturuluyor.
    user_movie_df = common_movies.pivot_table(index=["userId"], columns=["title"], values="rating")
    return user_movie_df

# Kullanıcı-film matrisi oluşturuluyor.
user_movie_df = create_user_movie_df()

def user_based_recommender(random_user, user_movie_df, ratio=60, cor_th=0.65, score=3.5):
    """
    Kullanıcı-temelli işbirlikçi filtreleme öneri fonksiyonu.
    
    Args:
        random_user (int): Öneri yapılacak kullanıcı ID'si.
        user_movie_df (pd.DataFrame): Kullanıcı-film matrisi.
        ratio (int, optional): İzlenen filmlerin yüzde kaçı kadar ortak film aransın. Default 60.
        cor_th (float, optional): Korelasyon eşik değeri. Default 0.65.
        score (float, optional): Önerilecek filmler için min ağırlıklı skor. Default 3.5.

    Returns:
        pd.DataFrame: Önerilen filmler ve ağırlıklı skorları ile isimleri.
    """
    import pandas as pd

    # random_user'ın puanladığı filmler alınır.
    random_user_df = user_movie_df[user_movie_df.index == random_user]
    movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()
    # Diğer kullanıcıların bu filmlere verdiği puanlardan bir df oluşturulur.
    movies_watched_df = user_movie_df[movies_watched]
    # Kullanıcıların kaç ortak film puanladığı hesaplanır.
    user_movie_count = movies_watched_df.T.notnull().sum()
    user_movie_count = user_movie_count.reset_index()
    user_movie_count.columns = ["userId", "movie_count"]
    # Filtreleme için eşik (perc) belirlenir.
    perc = len(movies_watched) * ratio / 100
    users_same_movies = user_movie_count[user_movie_count["movie_count"] > perc]["userId"]

    # Sadece ortak filmleri yeterli oranda izleyen kullanıcılar ve random_user'ın df'i birleştirilir.
    final_df = pd.concat([movies_watched_df[movies_watched_df.index.isin(users_same_movies)],
                          random_user_df[movies_watched]])

    # Korelasyon matrisi hesaplanır.
    corr_df = final_df.T.corr().unstack().sort_values().drop_duplicates()
    corr_df = pd.DataFrame(corr_df, columns=["corr"])
    corr_df.index.names = ['user_id_1', 'user_id_2']
    corr_df = corr_df.reset_index()

    # Yüksek korelasyona sahip kullanıcılar seçilir.
    top_users = corr_df[(corr_df["user_id_1"] == random_user) & (corr_df["corr"] >= cor_th)][
        ["user_id_2", "corr"]].reset_index(drop=True)

    # Korelasyona göre azalan sıralama yapılır ve sütun adı uyarlanır.
    top_users = top_users.sort_values(by='corr', ascending=False)
    top_users.rename(columns={"user_id_2": "userId"}, inplace=True)

    # Derecelendirme verisi yüklenir.
    rating = pd.read_csv('datasets/movie_lens_dataset/rating.csv')
    # Top users ile rating datası birleştirilir.
    top_users_ratings = top_users.merge(rating[["userId", "movieId", "rating"]], how='inner')
    # Ağırlıklı derecelendirme skoru hesaplanır.
    top_users_ratings['weighted_rating'] = top_users_ratings['corr'] * top_users_ratings['rating']

    # Her film için ağırlıklı ortalama hesaplanır.
    recommendation_df = top_users_ratings.groupby('movieId').agg({"weighted_rating": "mean"})
    recommendation_df = recommendation_df.reset_index()

    # Belirlenmiş score'dan büyük olan filmler seçilir ve ağırlıklı skora göre sıralanır.
    movies_to_be_recommend = recommendation_df[recommendation_df["weighted_rating"] > score].sort_values("weighted_rating", ascending=False)
    # Filmlere başlık eklenmek üzere movie.csv okunur.
    movie = pd.read_csv('datasets/movie_lens_dataset/movie.csv')
    # Önerilen filmler movieId ve başlık ile birlikte dönülür.
    return movies_to_be_recommend.merge(movie[["movieId", "title"]])

# Rasgele bir kullanıcı seçerek fonksiyon deneniyor.
random_user = int(pd.Series(user_movie_df.index).sample(1).values)
user_based_recommender(random_user, user_movie_df, cor_th=0.70, score=4)


,movieId,weighted_rating,title
